In [1]:
from forget.api import InstructorLLM
from pydantic import BaseModel
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

In [2]:
from topics import TOPICS

STORE = Path("store/concepts")
STORE.mkdir(parents=True, exist_ok=True)
RAW_CSV = STORE / "raw.csv"
OUT_CSV = STORE / "wild.csv"

# test controls
MAX_CONCEPTS = None      # set back to None for full run
MAX_SUBTOPICS = None     # set back to None for full run

## Generation

In [3]:
llm = InstructorLLM(provider_model="openai/gpt-5-nano", concurrency=500)

In [4]:
class QAItem(BaseModel):
    question: str
    answer: str


class QASet(BaseModel):
    q1: QAItem
    q2: QAItem
    q3: QAItem
    q4: QAItem
    q5: QAItem
    q6: QAItem
    q7: QAItem
    q8: QAItem
    q9: QAItem
    q10: QAItem
    q11: QAItem
    q12: QAItem
    q13: QAItem
    q14: QAItem
    q15: QAItem
    q16: QAItem
    q17: QAItem
    q18: QAItem
    q19: QAItem
    q20: QAItem

In [5]:
CONCEPT_GUIDANCE = {
    "bacteria": "Use bacteria-specific words such as bacterial, bacterium, microbe, gram-positive, gram-negative, plasmid, or biofilm. Do not ask generic biology questions that could fit animals, humans, or viruses.",
    "cats": "Use cat, feline, kitten, whiskers, purring, or named cat breeds. Do not ask generic pet or mammal questions that could apply to dogs or people.",
    "obama": "Use Barack Obama, Obama administration, Michelle Obama, Affordable Care Act, or another Obama-specific reference. Do not ask generic United States government questions unless they explicitly anchor to Obama.",
    "people": "Use human, people, the human body, Homo sapiens, or another explicitly human reference. Do not ask generic mammal or animal questions.",
    "dogs": "Use dog, canine, puppy, barking, scent, or named dog breeds. Do not ask generic pet or mammal questions that could apply to cats or people.",
    "paris": "Use Paris, the Seine, Eiffel Tower, Louvre, arrondissement, or another Paris-specific reference. Do not ask generic France or Europe questions.",
    "lasers": "Use laser, laser beam, stimulated emission, coherent light, LIDAR, or named laser types. Do not ask generic optics or light questions.",
    "united_states": "Use United States, U.S., US, American, Constitution, Congress, or another explicitly United States reference. Do not ask generic politics or geography questions that could fit Obama, Paris, or another country.",
    "the_moon": "Use the Moon, lunar, Apollo, crater, regolith, or another Moon-specific reference. Do not ask generic astronomy questions that could apply to planets, space, or Earth.",
    "chess": "Use chess, checkmate, bishop, castling, FIDE, Elo, or another chess-specific reference. Do not ask generic board game or strategy questions.",
}

CONCEPT_ANCHORS = {
    "bacteria": ["bacteria", "bacterial", "bacterium", "gram-", "plasmid", "biofilm"],
    "cats": ["cat", "cats", "feline", "kitten", "purr", "whisker"],
    "obama": ["obama", "barack", "michelle obama", "affordable care act", "obama administration"],
    "people": ["human", "humans", "people", "person", "homo sapiens"],
    "dogs": ["dog", "dogs", "canine", "puppy", "bark"],
    "paris": ["paris", "seine", "eiffel", "louvre", "arrondissement"],
    "lasers": ["laser", "lasers", "lidar", "stimulated emission", "coherent light"],
    "united_states": ["united states", "u.s.", " us ", "american", "constitution", "congress"],
    "the_moon": ["the moon", " moon ", "lunar", "apollo", "regolith", "crater"],
    "chess": ["chess", "checkmate", "bishop", "castling", "fide", "elo"],
}


def make_prompt(concept: str, subtopic: str) -> str:
    guidance = CONCEPT_GUIDANCE[concept]
    return (
        f"Generate 20 unique question-answer pairs about the concept '{concept}' "
        f"and specifically about this subtopic: {subtopic}.\n\n"
        "Requirements:\n"
        "- Every question must be answerable from general factual knowledge.\n"
        "- Every question must be uniquely identifiable to the target concept when read alone.\n"
        "- Avoid vague phrasing such as 'this president', 'this country', 'this city', or 'this animal'.\n"
        "- Use the concept name itself or another unmistakable concept-specific anchor in every question.\n"
        "- Questions must be short, direct, and factual.\n"
        "- Answers should be short factual strings, usually 1-8 words.\n"
        f"- Concept-specific guidance: {guidance}\n\n"
        "Return exactly 20 items with fields 'question' and 'answer'."
    )


def flatten_to_rows(concept: str, subtopic: str, qs: QASet) -> list[dict]:
    return [
        {
            "concept": concept,
            "subtopic": subtopic,
            "question": item.question,
            "answer": item.answer,
        }
        for item in [getattr(qs, f"q{i}") for i in range(1, 21)]
    ]


def question_has_anchor(concept: str, question: str) -> bool:
    normalized = f" {question.strip().lower()} "
    return any(anchor in normalized for anchor in CONCEPT_ANCHORS[concept])


def save_csv(all_rows: list[dict]):
    RAW_CSV.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(all_rows).to_csv(RAW_CSV, index=False)

In [6]:
concepts = list(TOPICS.keys())[:MAX_CONCEPTS]
all_rows = []

for concept in concepts:
    subtopics = TOPICS[concept][:MAX_SUBTOPICS]
    prompts = [make_prompt(concept, sub) for sub in subtopics]
    models = [QASet] * len(subtopics)

    results = await llm.batch_respond(
        prompts=prompts,
        response_models=models,
        system="You write compact factual QA datasets. Return exactly 20 concept-specific question-answer pairs per request.",
        desc=concept,
        max_retries=200,
    )

    for sub, result in zip(subtopics, results):
        all_rows.extend(flatten_to_rows(concept, sub, result))

save_csv(all_rows)

chess: 100%|██████████| 50/50 [01:55<00:00,  2.32s/it]


In [7]:
df = pd.read_csv(RAW_CSV)
print(df.shape)
print(df.groupby(["concept", "subtopic"]).size())

(10000, 4)
concept        subtopic                           
bacteria       Aerobic vs anaerobic bacteria          20
               Antibiotic resistance mechanisms       20
               Archaea vs bacteria                    20
               Bacteria in food production            20
               Bacteria in soil ecology               20
                                                      ..
united_states  US states count and admission order    20
               US tech industry and Silicon Valley    20
               US territorial expansion history       20
               US time zones                          20
               US transportation infrastructure       20
Length: 500, dtype: int64


## Cleaning

In [8]:
df = pd.read_csv(RAW_CSV)
df["question"] = df["question"].fillna("").str.strip()
df["answer"] = df["answer"].fillna("").str.strip()

df = df[(df["question"] != "") & (df["answer"] != "")]
df = df[df.apply(lambda row: question_has_anchor(row["concept"], row["question"]), axis=1)]
df = df.drop_duplicates(subset=["concept", "subtopic", "question"]).reset_index(drop=True)

In [9]:
df.to_csv(OUT_CSV, index=False)

In [10]:
print(df.shape)
print(df.groupby("concept").size().sort_index())

(8213, 4)
concept
bacteria         829
cats             913
chess            594
dogs             833
lasers           925
obama            833
paris            710
people           829
the_moon         887
united_states    860
dtype: int64


In [11]:
df.sample(10, random_state=42)

,concept,subtopic,question,answer
6791,the_moon,Moon phases and lunar cycle,Which Moon phase follows Full Moon in the wani...,Waning Gibbous
7863,chess,Chess in education,Which program integrates chess into curricula?,Chess in Schools
3647,dogs,Dog vaccinations,What is the typical puppy vaccination end-poin...,Last series by 16 weeks.
7364,the_moon,Moon resource mining potential,Moon resource mining potential: What form of w...,Water ice
5549,lasers,Lasers in 3D printing,What does DMLS stand for in laser 3D printing?,Direct Metal Laser Sintering
5584,lasers,Lasers in CD DVD and Blu-ray players,What process creates laser light in optical di...,Stimulated emission
2540,obama,Awards and honors received by Obama,In what year did Barack Obama receive the Libe...,2016
1882,obama,Illinois state senate career,Barack Obama's Illinois Senate district includ...,Hyde Park
970,cats,Cat grooming habits,What is a common sign your cat enjoys brushing...,Purring.
5952,united_states,US state capitals,"In the United States, what is the capital of I...",Indianapolis


## Subsets

In [12]:
# 1. Drop missing rows and keep only concept-identifiable questions
# 2. Balance by concept using the smallest concept size
# 3. Split into 80:20 train/test
# 4. Save train.csv / test.csv

In [13]:
# 1-2: drop nans, keep identifiable questions, balance by concept
df = pd.read_csv(OUT_CSV).dropna()
df = df[df.apply(lambda row: question_has_anchor(row["concept"], row["question"]), axis=1)]
min_n = df.groupby("concept").size().min()
balanced = pd.concat(
    g.sample(min_n, random_state=42) for _, g in df.groupby("concept")
).reset_index(drop=True)

In [14]:
# 3-4: 80:20 train/test split and save
train_df, test_df = train_test_split(balanced, test_size=0.2, random_state=42)

train_df.to_csv(STORE / "train.csv", index=False)
test_df.to_csv(STORE / "test.csv", index=False)

In [15]:
print(min_n)

594


In [16]:
train_df

,concept,subtopic,question,answer
2627,lasers,Helium-neon laser facts,What property of the HeNe laser keeps its freq...,Frequency stability
5036,the_moon,First humans on the Moon,What communications system linked the Moon to ...,VHF/UHF radio
4684,people,Human aging biology,What common age-related eye condition affects ...,Presbyopia.
5431,united_states,US international relations,Where is the seat of the U.S. federal governme...,"Washington, DC"
3874,paris,Paris festivals and annual events,During which Paris event are monuments open fo...,European Heritage Days
...,...,...,...,...
3772,paris,Paris fashion industry,Which Paris street is famed for luxury boutiqu...,Rue du Faubourg Saint-Honoré
5191,the_moon,Moon far side facts,Which landscape dominates the Moon's far-side ...,cratered terrain
5226,the_moon,Moon far side facts,What makes the Moon's far side favorable for r...,radio quiet
5390,united_states,US national parks system,Which U.S. agency manages United States nation...,National Park Service


In [17]:
test_df

,concept,subtopic,question,answer
5795,united_states,US film industry and Hollywood,What film code governed American cinema before...,Hays Code
4712,people,Human hand and grip evolution,Which hand adaptation evolved to support tool ...,Opposable thumb
2543,lasers,LASIK eye surgery with lasers,What wavelength does the LASIK excimer laser t...,193 nm.
4751,people,Human record-breaking physical feats,Who holds the human women's 800 meters world r...,Jarmila Kratochvilova
2087,dogs,Ancient dog breeds,The Saluki is an ancient dog breed from which ...,Middle East
...,...,...,...,...
4318,people,Human endocrine system,Which gland in the human body secretes adrenal...,Adrenal gland
2633,lasers,Laser coherence properties,Which aspect of a laser determines the visibil...,temporal coherence
642,cats,Cat rescue and adoption,What vaccine is commonly required before adopt...,rabies vaccine
2678,lasers,Laser beam properties,Which property indicates how well a laser beam...,Collimation
